In [1]:
# folder with text files
folder_with_text = './text'

# max text length
max_length=1024

# latent space z size
internal_dim=32

# training batch size
batch_size=128

# where to save model checkpoints
checkpoint_path = 'runs/test-autoregressive-gd2-padmask'

# number of epochs to train
num_epochs=500

# Create tokenizer

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch import Tensor
from typing import List, Dict
import os
from kemsekov_torch.text_tools import SimpleTokenizer


txt_files = [[os.path.join(fdir,f) for f in files if f.endswith(".txt")] for fdir,_,files in os.walk(folder_with_text)]
txt_files = [b for a in txt_files for b in a]
txt_lines = [open(v).read() for v in txt_files]

tokenizer = SimpleTokenizer(txt_lines,lowercase=True,unknown_symbols_placeholder=' ')
torch.jit.script(tokenizer).save("tokenizer.pt")

test_str="This is my TEST string! Раз!"
inds=tokenizer.encode(test_str)
print(test_str)
print(inds)
print(tokenizer.decode(inds))

Text length analysis
text lines	 79295
line chars mean	 78.267
line chars std	 180.905
0.05 quantile	 0.0
0.95 quantile	 341.0
0.995 quantile	 711.0
This is my TEST string! Раз!
tensor([54, 42, 43, 53,  1, 43, 53,  1, 47, 59,  1, 54, 39, 53, 54,  1, 53, 54,
        52, 43, 48, 41,  2,  1,  1,  1,  1,  2])
this is my test string!    !


/home/bochkarev/Programs/venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


# Define Dataset

In [3]:
import math
from kemsekov_torch.train import split_dataset
import torch
from kemsekov_torch.text_tools import TokenDataset

txt_split = [b for v in txt_lines if len(v)>30 for b in v.split('\n')]
dataset = TokenDataset(
    tokenizer,
    txt_split[:15000],
    pad_token=tokenizer.unknown_symbols_placeholder,
    batch_size=batch_size,
    fixed_length=max_length
)

# split dataset into train and test
train_dataset,test_dataset,train_loader, test_loader = split_dataset(
    dataset,
    test_size=0.05,
    batch_size=batch_size,
    random_state=None,
    # bin_by_size=True,
    num_workers=1,
)

Train items 6785
Test items 358


In [4]:
import random
ind = random.randint(0,len(train_dataset)-1)
inds = dataset[ind]

print("Text length",len(inds))
skip_text = tokenizer.decode(inds).strip()
print(skip_text)

for t in train_loader: break
print("batch size sample",t.shape)

Text length 1024
“oh, yes he would,” said ron, even more loudly than dean.


batch size sample torch.Size([128, 1024])


# Define Model

In [5]:
from typing import Literal

#both implement re-zero approach
from kemsekov_torch.attention import SelfAttention
from kemsekov_torch.gated_delta_2 import GatedDelta2Scan

from kemsekov_torch.common_modules import Residual, Transpose, SwiGLU,ConcatTensors,SumTensors
import torch
import torch.nn as nn

# module to convert text tokens to vector
class Embedding(nn.Module):
    """
    Module for token to embedding vector learning
    """
    def __init__(self, vocab_size, embedding_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_size = embedding_size

        # Initialize weights and bias
        self.weight = nn.Parameter(torch.Tensor(vocab_size, embedding_size))
        self.bias = nn.Parameter(torch.Tensor(embedding_size))

        self.reset_parameters()

    #normal init
    def reset_parameters(self):
        # Initialize weights with a normal distribution
        std = 1.0 / (self.vocab_size**0.5)
        nn.init.normal_(self.weight, mean=0.0, std=std)
        # Initialize bias to zeros
        nn.init.zeros_(self.bias)
        
    def forward(self, input):
        # Input is expected to be a tensor of indices
        output = torch.nn.functional.embedding(input, self.weight) + self.bias
        return output

class AutoregressiveChar(nn.Module):
    def __init__(
        self,
        vocab_size,
        internal_dim,
        layers=3,
        mlp_factor=4,
        heads=8,
        impl:Literal['attn','gd2']="attn",
        merge_implementation:Literal['concat','sum']='sum'
    ):
        super().__init__()

        def mlp():
            return Residual([
                nn.RMSNorm(internal_dim),
                SwiGLU(internal_dim,internal_dim*mlp_factor),
                nn.Linear(internal_dim*mlp_factor,internal_dim),
            ])
            
        def get_imp():
            if impl=='attn':
                return nn.Sequential(
                    Transpose(1,-1),
                    SelfAttention(
                        internal_dim,
                        heads=heads,
                        kv_heads=heads//2,
                        head_dim=64,
                        add_absolute_pos=True,
                        prenorm='rms',
                        is_causal=True,
                        dimensions=1
                    ),
                    Transpose(1,-1),
                    mlp()
                )
            if impl=='gd2':
                return nn.Sequential(
                    GatedDelta2Scan(
                        dim=internal_dim,
                        heads=heads,
                        kv_heads=heads//2,
                        QK_dim=64,
                        V_dim=64
                    ),
                    mlp()
                )
        self.encode = Embedding(
            vocab_size,embedding_size=internal_dim
        )
        
        if merge_implementation=='concat':
            self.merge_activations=nn.Sequential(
                ConcatTensors(-1),
                nn.Linear(internal_dim*2,internal_dim,bias=False)
            )
        if merge_implementation=='sum':
            self.merge_activations=SumTensors()
        
        self.middle=nn.Sequential(*[
            get_imp()
            for i in range(layers)
        ])
        self.decode=nn.Linear(
            internal_dim,vocab_size,bias=False
        )
        
    def forward(self,ind,previous_activations = None):
        x = self.encode(ind)
        previous_activations=x*0 if previous_activations is None else previous_activations
        
        x=self.merge_activations([x,previous_activations])
        x = self.middle(x)
        return x,self.decode(x)
    
    def params_count(self):
        return sum([p.numel() for p in self.parameters()])

#how to use this model
#step1: encode input ids
#step2: 

model = AutoregressiveChar(tokenizer.vocab_size,256,layers=1,mlp_factor=1,impl='gd2')
print(model.params_count())
[c.shape for c in model(t[:,:128])]

829442


[torch.Size([128, 128, 256]), torch.Size([128, 128, 80])]

# Training

In [6]:
from kemsekov_torch.train import train
from kemsekov_torch.metrics import f1_score
from accelerate.utils import TorchDynamoPlugin
from torchmetrics.classification import MulticlassF1Score

# Initialize the metric object
f1_metric = MulticlassF1Score(num_classes=tokenizer.vocab_size, average='macro').cuda()

CE = torch.nn.CrossEntropyLoss()


def get_pad_mask(next_t: torch.Tensor, pad_token: int) -> torch.Tensor:
    """
    Finds the first occurrence of 3 sequential pad tokens per batch row 
    and returns a flattened boolean mask of shape [BATCH * seqlen].
    Elements at and after the 3 pads are set to False.
    
    We use this thing to compute loss on non-padded part of batch
    """
    is_pad = (next_t == pad_token)
    
    # Check 3 sequential tokens using slicing
    sequential_3_pads = is_pad[:, :-2] & is_pad[:, 1:-1] & is_pad[:, 2:]
    
    # Find the first index along dim=1 where this happens per batch item
    has_3_pads = sequential_3_pads.any(dim=-1, keepdim=True)
    first_pad_idx = torch.argmax(sequential_3_pads.int(), dim=-1, keepdim=True)
    
    # Create index grid to build the mask
    seq_indices = torch.arange(next_t.shape[1], device=next_t.device).unsqueeze(0)
    
    # Retain elements before the 3-pad boundary
    mask = torch.ones_like(next_t, dtype=torch.bool)
    mask = torch.where(has_3_pads, seq_indices < first_pad_idx, mask)
    
    return mask

def compute_loss_and_metric(model,batch):
    prev_t = batch[:,:-1]
    next_t = batch[:,1:]
    activations,logits = model(prev_t)
    mask = get_pad_mask(next_t, pad_token=dataset.pad_token[0]).flatten()
    
    logits=logits.view(-1,logits.shape[-1])[mask]
    next_t=next_t.flatten()[mask]
    
    loss = CE(logits,next_t)
    f1 = f1_metric(logits,next_t)
    return loss,{
        'f1':f1
    }

_ = train(
    model,
    train_loader,
    test_loader,
    compute_loss_and_metric,
    checkpoint_path,
    # f'{checkpoint_path}/last',
    gradient_clipping_max_norm=1,
    accelerate_args=dict(
        mixed_precision='fp16',
        dynamo_plugin = TorchDynamoPlugin(
            backend="inductor",
            mode="default",
            fullgraph=False,
            dynamic=True          # Enables torch.compile(dynamic=True)
        )
    ),
    save_on_metric_improve=['f1'],
    num_epochs=100,
    checkpoints_count=1,
    # default_lr=0.01
)

/home/bochkarev/Programs/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using dir runs/test-autoregressive-gd2-padmask
Using default fused AdamW optimizer
Using default CosineAnelingScheduler
Total model parameters 0.83 M
Using device cuda

Epoch 1/100


train 0: 100%|██████████| 53/53 [00:14<00:00,  3.57it/s, f1=0.1449, loss=2.0275]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 2.42423 | 2.00377 |
|  f1  | 0.1051  | 0.1682  |
+------+---------+---------+
saved epoch-1

Epoch 2/100


train 0: 100%|██████████| 53/53 [00:06<00:00,  7.72it/s, f1=0.1994, loss=1.7990]


...

+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 1.06191 | 1.23209 |
|  f1  | 0.4445  | 0.3922  |
+------+---------+---------+

Epoch 99/100


train 0: 100%|██████████| 53/53 [00:07<00:00,  7.52it/s, f1=0.4149, loss=1.0876]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 1.06173 | 1.23213 |
|  f1  | 0.4446  | 0.3922  |
+------+---------+---------+

Epoch 100/100


train 0: 100%|██████████| 53/53 [00:07<00:00,  7.50it/s, f1=0.4142, loss=1.0876]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 1.06164 | 1.23212 |
|  f1  | 0.4446  | 0.3922  |
+------+---------+---------+
Empty cuda cache complete


In [7]:
import torch
import torch.nn.functional as F

def sample(logits: torch.Tensor, temp: float = 0.7, top_p: float = 0.9) -> torch.Tensor:
    """
    Samples class indices from a batch of logits using temperature scaling and Top-P (Nucleus) filtering.
    
    Args:
        logits (torch.Tensor): Tensor of shape [BATCH, classes]
        temp (float): Temperature for scaling. 0.0 performs greedy argmax.
        top_p (float): Nucleus sampling threshold (0.0 < top_p <= 1.0). 
                       1.0 turns off Top-P filtering.
    """
    # 1. Handle greedy selection if temperature is 0
    if temp == 0.0:
        return torch.argmax(logits, dim=-1)
        
    # 2. Scale the logits by the temperature
    scaled_logits = logits / temp
    
    # 3. Apply Top-P (Nucleus) filtering if top_p < 1.0
    if top_p < 1.0:
        # Sort probabilities/logits in descending order
        sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True, dim=-1)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        
        # Calculate cumulative probabilities
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        
        # Remove tokens with cumulative probability above the threshold
        # We shift the mask by 1 to make sure we keep the first token that exceeds top_p
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = False
        
        # Mask out excluded logits by setting them to negative infinity
        # This gives them 0 probability during the final softmax step
        sorted_logits[sorted_indices_to_remove] = float('-inf')
        
        # Scatter the filtered logits back to their original position mapping
        scaled_logits = torch.gather(sorted_logits, dim=-1, index=sorted_indices.argsort(dim=-1))

    # 4. Final softmax and multinomial sampling
    probs = F.softmax(scaled_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)

In [9]:
start = "Some days ago"
ids = tokenizer.encode(start).tolist()
model.eval()

to_generate=1024

for i in range(to_generate):
    with torch.no_grad():
        act,logits = model(torch.tensor([ids],device='cuda'))
        next_token = sample(logits[:,-1],temp=0.7).item()
        ids.append(next_token)
print(tokenizer.decode(ids))

some days ago, the wall down at the chamber of parchment, the tournament, watching her club, and a girl who was wearing a traditional seemed to be party. the skrewts, and they had to the size of a dragon the parchment they were she was a girl wing a half fire on the car to the hogwarts school concentrate. the rest of days and dived eased to be in his fingers from a moment than the ground him that he had appeared the same to resembled fervently and durmstrang to stay at a few moment of his chair behind the champions. he was an any looked stands, and it for a split was concerned the tones will be done beauxbatons and stared to teach the staircase, and were looking up the castle when in the familiar fellow and bludger that madam pomfrey into the door. they read and said to talk to his face for the stone of the rest of way to the great looked stupidled off the second of the school had been caught with the door scarlet in the end of the screaming and leaving a wailing the stared at her. he 